In [ ]:
# ============================================================
# UDA PIPELINE 
# NCT-CRC-HE-100K (Source, Labeled) → TCGA (Target, Unlabeled)
# Backbone: UNI2-H (Frozen)
# Trainable: Adapter + Classifier Head
#
# Ablation (Option A / paper-consistent):
#   B0 = CE (baseline)
#   B1 = CE + CORAL
#   B2 = CE + CORAL + MKMMD
#   B3 = CE + CORAL + MKMMD + Proto
#   B4 = CE + CORAL + MKMMD + Proto + Stain-consistency
#
# Eval: CRC-VAL-HE-7K (Macro-F1, BalancedAcc, Acc) + TCGA Alignment Proof
# Add-ons:
#   - Best checkpoint (UDA score) + Early Stopping
#   - Ramp-up for alignment/proto terms
#   - NOTEBOOK SAFE: num_workers_train=0 by default to avoid spawn/pickle errors
#   - UMAP before/after using same projector
# ============================================================

import os, glob, random, time, csv, math, subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import umap.umap_ as umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

try:
    import psutil
except ImportError:
    psutil = None

# Optional: paired t-test p-value (preferred)
try:
    from scipy.stats import ttest_rel
except Exception:
    ttest_rel = None

# -------------------------
# Reproducibility helpers
# -------------------------
SEED = 42  # default / fallback seed (main loop sets per-seed)
torch.backends.cudnn.benchmark = True  # keep non-deterministic for speed as requested

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 1) PATHS
# =========================
NCT_PATH_100K = "crc_data/dataset/train"
NCT_PATH_7K   = "crc_data/dataset/val"
TCGA_PATH     = "tcga_download_with_aws/patches/all"

# --- NEW: ablation/final output separation ---
EXPERIMENT_PROFILE = "ablation"  # "ablation" or "final"

OUT_BASE = "uda_outputs"
OUT_DIR = os.path.join(OUT_BASE, EXPERIMENT_PROFILE)
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 2) PREPROCESS
# =========================
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

def normalize_imnet(x01):
    return (x01 - IMNET_MEAN.to(x01.device)) / IMNET_STD.to(x01.device)

def resize_center_crop_pil(img, out_size=224):
    img = img.resize((256,256), resample=Image.BILINEAR)
    left = (256 - out_size) // 2
    top  = (256 - out_size) // 2
    return img.crop((left, top, left + out_size, top + out_size))

def pil_to_tensor01(img):
    arr = np.array(img).astype(np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2,0,1)  # HWC -> CHW
    return t

def stain_perturb(x01):
    """
    Minimal stain/illumination perturbation (pre-normalization).
    x01: (B,3,H,W) in [0,1]
    """
    noise = 0.03 * torch.randn_like(x01)
    scale = (0.9 + 0.2 * torch.rand(x01.size(0), 1, 1, 1, device=x01.device))
    x_aug = torch.clamp(x01 * scale + noise, 0.0, 1.0)
    return x_aug

# =========================
# 3) DATASETS
# =========================
def list_images(root):
    exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")
    paths = []
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(exts):
                paths.append(os.path.join(r, f))
    return paths

class NCTFolderDataset(Dataset):
    """
    root/
      ADI/*.png
      BACK/*.png
      ...
    label = folder name
    returns: x01 (3,224,224) in [0,1], y int, path str
    """
    def __init__(self, root, class_to_idx=None, out_size=224):
        self.root = root
        self.out_size = out_size

        ignore = {".ipynb_checkpoints", "__pycache__", ".DS_Store"}
        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
            and d not in ignore
            and not d.startswith(".")
        ])

        assert len(self.classes) == 9, f"Expected 9 classes, found {len(self.classes)}: {self.classes}"

        if class_to_idx is None:
            self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        else:
            self.class_to_idx = class_to_idx
            assert set(self.classes) == set(self.class_to_idx.keys()), (
                f"Val classes mismatch.\nVal: {self.classes}\nTrain keys: {sorted(self.class_to_idx.keys())}"
            )

        self.samples = []
        for c in self.classes:
            cdir = os.path.join(root, c)
            for p in list_images(cdir):
                self.samples.append((p, self.class_to_idx[c]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = resize_center_crop_pil(img, self.out_size)
        x01 = pil_to_tensor01(img)
        return x01, y, path

class TCGAUnlabeledDataset(Dataset):
    """
    supports fixed_paths so TCGA sample set is identical across seeds/runs.
    """
    def __init__(self, root, out_size=224, max_samples=None, seed=42, fixed_paths=None):
        self.root = root
        self.out_size = out_size

        if fixed_paths is not None:
            self.paths = list(fixed_paths)
        else:
            self.paths = []
            for p in glob.glob(os.path.join(root, "*")):
                if p.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp")):
                    self.paths.append(p)
            self.paths = sorted(self.paths)

            if max_samples is not None and len(self.paths) > max_samples:
                rng = np.random.default_rng(seed)
                self.paths = list(rng.choice(self.paths, size=max_samples, replace=False))

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        img = resize_center_crop_pil(img, self.out_size)
        x01 = pil_to_tensor01(img)
        return x01, path

def make_fixed_tcga_paths(root, max_samples=None, seed=42):
    """
    Deterministic TCGA file list:
      - sorted file list
      - optional subsample with fixed seed (DATA seed, not TRAIN seed)
    """
    paths = []
    for p in glob.glob(os.path.join(root, "*")):
        if p.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp")):
            paths.append(p)
    paths = sorted(paths)

    if max_samples is not None and len(paths) > max_samples:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(paths), size=max_samples, replace=False)
        paths = [paths[i] for i in sorted(idx)]
    return paths

def make_loaders(batch_size=64, num_workers_train=0, num_workers_eval=0, tcga_max_samples=None, fixed_tcga_paths=None):
    """
    NOTEBOOK SAFE DEFAULT:
      num_workers_train=0 avoids spawn/pickle errors.
    """
    train_ds = NCTFolderDataset(NCT_PATH_100K, class_to_idx=None, out_size=224)
    val_ds   = NCTFolderDataset(NCT_PATH_7K,   class_to_idx=train_ds.class_to_idx, out_size=224)
    assert train_ds.classes == val_ds.classes, "Train/Val class mismatch!"

    # IMPORTANT: tcga_max_samples should be None here if fixed_tcga_paths is provided.
    tcga_ds  = TCGAUnlabeledDataset(
        TCGA_PATH, out_size=224, max_samples=tcga_max_samples, seed=SEED,
        fixed_paths=fixed_tcga_paths
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers_train, pin_memory=True, drop_last=True,
        persistent_workers=(num_workers_train > 0)
    )

    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers_eval, pin_memory=True, drop_last=False
    )

    tcga_loader = DataLoader(
        tcga_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers_train, pin_memory=True, drop_last=True,
        persistent_workers=(num_workers_train > 0)
    )

    return train_loader, val_loader, tcga_loader, train_ds.classes

# =========================
# 4) MODEL
# =========================
def load_uni2h_encoder():
    timm_kwargs = {
        'img_size': 224, 'patch_size': 14, 'depth': 24, 'num_heads': 24,
        'init_values': 1e-5, 'embed_dim': 1536, 'mlp_ratio': 2.66667 * 2,
        'num_classes': 0, 'no_embed_class': True,
        'mlp_layer': timm.layers.SwiGLUPacked, 'act_layer': torch.nn.SiLU,
        'reg_tokens': 8, 'dynamic_img_size': True
    }
    enc = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
    return enc

class BottleneckAdapter(nn.Module):
    def __init__(self, dim=1536, reduction=16):
        super().__init__()
        h = max(dim // reduction, 64)
        self.net = nn.Sequential(
            nn.Linear(dim, h),
            nn.GELU(),
            nn.Linear(h, dim),
        )
    def forward(self, x):
        return x + self.net(x)

class LinearHead(nn.Module):
    def __init__(self, in_dim=1536, num_classes=9):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
    def forward(self, x): return self.fc(x)

# =========================
# 5) LOSS COMPONENTS
# =========================
def coral_loss(fs, ft):
    fs = fs - fs.mean(dim=0, keepdim=True)
    ft = ft - ft.mean(dim=0, keepdim=True)
    cov_s = (fs.t() @ fs) / (fs.size(0) - 1)
    cov_t = (ft.t() @ ft) / (ft.size(0) - 1)
    return ((cov_s - cov_t) ** 2).mean()

def _gaussian_kernel(x, y, sigma):
    x_norm = (x**2).sum(dim=1, keepdim=True)
    y_norm = (y**2).sum(dim=1, keepdim=True)
    dist = x_norm - 2.0 * (x @ y.t()) + y_norm.t()
    return torch.exp(-dist / (2 * sigma * sigma))

def mkmmd_loss(fs, ft, sigmas=(1,2,4,8,16)):
    Kxx = 0; Kyy = 0; Kxy = 0
    for s in sigmas:
        Kxx = Kxx + _gaussian_kernel(fs, fs, s)
        Kyy = Kyy + _gaussian_kernel(ft, ft, s)
        Kxy = Kxy + _gaussian_kernel(fs, ft, s)
    return Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean()

@torch.no_grad()
def compute_source_prototypes(encoder, adapter, src_loader, device, num_classes, max_batches=200):
    encoder.eval(); adapter.eval()
    sums = torch.zeros(num_classes, 1536, device=device)
    cnts = torch.zeros(num_classes, device=device)

    for b, (x01, y, _) in enumerate(src_loader):
        if b >= max_batches: break
        x01 = x01.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)

        f = encoder(normalize_imnet(x01))
        if isinstance(f, (tuple, list)): f = f[0]
        f = adapter(f)

        for c in y.unique():
            c = int(c.item())
            m = (y == c)
            sums[c] += f[m].sum(dim=0)
            cnts[c] += m.sum()

    protos = sums / cnts.unsqueeze(1).clamp_min(1.0)
    return protos  # (C,D)

def proto_contrast_loss(fs, ys, protos, T=0.07):
    fs = F.normalize(fs, dim=1)
    p  = F.normalize(protos, dim=1)
    logits = (fs @ p.t()) / T
    return F.cross_entropy(logits, ys)

def stain_consistency_loss(encoder, adapter, xt01):
    x_aug01 = stain_perturb(xt01)
    f0 = encoder(normalize_imnet(xt01))
    f1 = encoder(normalize_imnet(x_aug01))
    if isinstance(f0, (tuple, list)): f0 = f0[0]
    if isinstance(f1, (tuple, list)): f1 = f1[0]
    f0 = adapter(f0)
    f1 = adapter(f1)
    return F.mse_loss(f0, f1)

# =========================
# 6) EXTRA METRICS (PAD + NC-Entropy) + RESOURCES
# =========================
def shannon_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log(p), axis=-1)

@torch.no_grad()
def compute_nc_entropy_and_majority_from_knn(src_feats, src_labels, tgt_feats, k=25, num_classes=9):
    """
    NC-Entropy: normalized Shannon entropy of kNN label distribution around each target point.
    majority: mean(max_c p(c)) across targets (higher = cleaner neighborhoods).
    """
    if torch.is_tensor(src_feats): src_feats = src_feats.detach().cpu()
    if torch.is_tensor(tgt_feats): tgt_feats = tgt_feats.detach().cpu()
    if torch.is_tensor(src_labels): src_labels = src_labels.detach().cpu()

    src_feats = F.normalize(torch.as_tensor(src_feats), dim=1)
    tgt_feats = F.normalize(torch.as_tensor(tgt_feats), dim=1)

    sim = tgt_feats @ src_feats.t()               # (Nt, Ns)
    nn_idx = torch.topk(sim, k=k, dim=1).indices  # (Nt, k)
    nn_lab = src_labels[nn_idx]                   # (Nt, k)

    Nt = nn_lab.size(0)
    C = int(num_classes)
    counts = torch.zeros(Nt, C, dtype=torch.float32)
    for c in range(C):
        counts[:, c] = (nn_lab == c).float().sum(dim=1)
    p = (counts / float(k)).numpy()

    maj = np.max(p, axis=1)
    majority_ratio_mean = float(np.mean(maj))

    H = shannon_entropy(p)
    Hnorm = H / np.log(C)  # normalize to [0,1]
    nc_entropy_mean = float(np.mean(Hnorm))
    return majority_ratio_mean, nc_entropy_mean

def compute_pad_proxy_a_distance(src_feats, tgt_feats, seed=42, max_samples=5000):
    """
    Proxy-A-distance (PAD) in the common form:
      PAD = 2 * (1 - 2 * eps), eps = test error of a domain classifier.
    Equivalent: PAD = 4*acc - 2 (since eps = 1-acc).
    """
    if torch.is_tensor(src_feats): src_feats = src_feats.detach().cpu().numpy()
    if torch.is_tensor(tgt_feats): tgt_feats = tgt_feats.detach().cpu().numpy()

    rng = np.random.default_rng(seed)
    Ns = src_feats.shape[0]
    Nt = tgt_feats.shape[0]
    ns = min(Ns, max_samples)
    nt = min(Nt, max_samples)

    src_idx = rng.choice(Ns, size=ns, replace=False) if Ns > ns else np.arange(Ns)
    tgt_idx = rng.choice(Nt, size=nt, replace=False) if Nt > nt else np.arange(Nt)

    X = np.vstack([src_feats[src_idx], tgt_feats[tgt_idx]])
    y = np.concatenate([np.zeros(len(src_idx)), np.ones(len(tgt_idx))])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )

    clf = LogisticRegression(
        max_iter=2000, n_jobs=-1, class_weight="balanced", random_state=seed
    )
    clf.fit(X_train, y_train)
    acc = float(clf.score(X_test, y_test))
    eps = 1.0 - acc
    pad = 2.0 * (1.0 - 2.0 * eps)
    return float(pad), float(eps), float(acc)

# --- NEW: nvidia-smi fallback snapshot (pynvml yokken GPU util 0 görünmesin diye) ---
def _nvidia_smi_snapshot(gpu_index=0):
    """
    Returns (util_gpu_pct, mem_used_mb, mem_total_mb) or (None, None, None) if unavailable.
    """
    try:
        cmd = [
            "nvidia-smi",
            f"--id={gpu_index}",
            "--query-gpu=utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits"
        ]
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True).strip()
        parts = [p.strip() for p in out.split(",")]
        if len(parts) >= 3:
            util = float(parts[0])
            mem_used = float(parts[1])
            mem_total = float(parts[2])
            return util, mem_used, mem_total
    except Exception:
        pass
    return None, None, None

def get_resource_snapshot(epoch_time_sec):
    max_gpu_gb = 0.0
    if torch.cuda.is_available():
        max_gpu_gb = torch.cuda.max_memory_allocated() / (1024**3)

    ram_gb = 0.0
    if psutil is not None:
        vm = psutil.virtual_memory()
        ram_gb = (vm.total - vm.available) / (1024**3)

    gpu_util = None
    gpu_mem_used_mb = None
    gpu_mem_total_mb = None
    if torch.cuda.is_available():
        util, mem_used, mem_total = _nvidia_smi_snapshot(gpu_index=0)
        gpu_util = util
        gpu_mem_used_mb = mem_used
        gpu_mem_total_mb = mem_total

    return float(epoch_time_sec), float(max_gpu_gb), float(ram_gb), gpu_util, gpu_mem_used_mb, gpu_mem_total_mb

# =========================
# 7) EVAL
# =========================
@torch.no_grad()
def eval_7k(encoder, adapter, head, val_loader, device):
    encoder.eval(); adapter.eval(); head.eval()
    y_true, y_pred = [], []
    for x01, y, _ in tqdm(val_loader, desc="Eval 7K", dynamic_ncols=True, mininterval=2):
        x01 = x01.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)
        f = encoder(normalize_imnet(x01))
        if isinstance(f, (tuple, list)): f = f[0]
        f = adapter(f)
        logits = head(f)
        pred = logits.argmax(dim=1)
        y_true.append(y.cpu().numpy())
        y_pred.append(pred.cpu().numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)
    return macro_f1, bal_acc, acc

@torch.no_grad()
def eval_tcga_alignment(
    encoder, adapter, src_loader, tgt_loader, device, num_classes, num_batches=50,
    knn_k=25, pad_max_samples=5000, pad_seed=42
):
    """
    Alignment proof metrics:
      - TCGA_CORAL
      - TCGA_MKMMD
      - TCGA_ProtoDist
      - TCGA_PAD (Proxy-A-distance)
      - TCGA_NCEntropy (normalized)
      - TCGA_Majority (mean majority ratio)
    """
    encoder.eval(); adapter.eval()
    protos = compute_source_prototypes(encoder, adapter, src_loader, device, num_classes=num_classes, max_batches=200)
    p = F.normalize(protos, dim=1)

    coral_vals, mkmmd_vals, pd_vals = [], [], []
    it_s = iter(src_loader)
    it_t = iter(tgt_loader)

    src_feat_bank, src_lab_bank, tgt_feat_bank = [], [], []

    for _ in range(num_batches):
        try:
            xs01, ys, _ = next(it_s)
        except StopIteration:
            it_s = iter(src_loader)
            xs01, ys, _ = next(it_s)

        try:
            xt01, _ = next(it_t)
        except StopIteration:
            it_t = iter(tgt_loader)
            xt01, _ = next(it_t)

        xs01 = xs01.to(device, non_blocking=True)
        ys   = ys.to(device, non_blocking=True)
        xt01 = xt01.to(device, non_blocking=True)

        fs = encoder(normalize_imnet(xs01))
        ft = encoder(normalize_imnet(xt01))
        if isinstance(fs, (tuple, list)): fs = fs[0]
        if isinstance(ft, (tuple, list)): ft = ft[0]
        fs = adapter(fs)
        ft = adapter(ft)

        coral_vals.append(float(coral_loss(fs, ft).item()))
        mkmmd_vals.append(float(mkmmd_loss(fs, ft).item()))

        ft_n = F.normalize(ft, dim=1)
        sim = ft_n @ p.t()
        proto_dist = (1.0 - sim.max(dim=1).values).mean()
        pd_vals.append(float(proto_dist.item()))

        src_feat_bank.append(fs.detach().cpu())
        src_lab_bank.append(ys.detach().cpu())
        tgt_feat_bank.append(ft.detach().cpu())

    coral_mean = float(np.mean(coral_vals))
    mkmmd_mean = float(np.mean(mkmmd_vals))
    pd_mean    = float(np.mean(pd_vals))

    src_bank = torch.cat(src_feat_bank, dim=0)
    tgt_bank = torch.cat(tgt_feat_bank, dim=0)
    src_labels = torch.cat(src_lab_bank, dim=0)

    pad, pad_eps, pad_acc = compute_pad_proxy_a_distance(
        src_bank, tgt_bank, seed=pad_seed, max_samples=pad_max_samples
    )
    majority, nc_entropy = compute_nc_entropy_and_majority_from_knn(
        src_bank, src_labels, tgt_bank, k=knn_k, num_classes=num_classes
    )

    return coral_mean, mkmmd_mean, pd_mean, pad, nc_entropy, majority

# =========================
# 8) VISUALIZATION
# =========================
def plot_loss_curves(history, out_path, title):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.figure(figsize=(10,6))
    plt.plot(history["total"], label="total")
    plt.plot(history["src"], label="src-cls")
    if np.max(history["proto"]) > 0: plt.plot(history["proto"], label="proto")
    if np.max(history["coral"]) > 0: plt.plot(history["coral"], label="coral")
    if np.max(history["mmd"]) > 0:   plt.plot(history["mmd"], label="mkmmd")
    if np.max(history["stain"]) > 0: plt.plot(history["stain"], label="stain-cons")
    plt.legend()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

def plot_epoch_loss_curve(epoch_means, out_path, title):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.figure(figsize=(8,5))
    plt.plot(range(1, len(epoch_means)+1), epoch_means, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Mean Train Loss")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

def show_plot_from_file(path, title=None, figsize=(8,6)):
    if not os.path.exists(path):
        print(f"Plot not found: {path}")
        return
    img = Image.open(path)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()

@torch.no_grad()
def extract_features_for_umap(encoder, adapter, loader, device, max_batches=40, return_labels=False):
    encoder.eval(); adapter.eval()
    feats = []
    labels = []
    for b, batch in enumerate(loader):
        if b >= max_batches: break

        x01 = batch[0].to(device, non_blocking=True)
        y = None
        if return_labels and len(batch) >= 2 and torch.is_tensor(batch[1]):
            y = batch[1]

        f = encoder(normalize_imnet(x01))
        if isinstance(f, (tuple, list)): f = f[0]
        f = adapter(f)
        feats.append(f.detach().cpu().numpy())

        if return_labels and y is not None:
            labels.append(y.detach().cpu().numpy())

    feats = np.vstack(feats)
    if return_labels:
        labels = np.concatenate(labels) if len(labels) > 0 else None
        return feats, labels
    return feats

def fit_umap_projector(src_feats, tgt_feats, pca_dim=30, seed=42):
    scaler = StandardScaler()
    pca = PCA(n_components=pca_dim, random_state=seed)
    um = umap.UMAP(n_components=2, random_state=seed)
    X = np.vstack([src_feats, tgt_feats])
    Xp = pca.fit_transform(scaler.fit_transform(X))
    um.fit(Xp)
    return scaler, pca, um

def project_umap(scaler, pca, um, feats):
    return um.transform(pca.transform(scaler.transform(feats)))

def plot_domain_gap(nct_2d, tcga_2d, out_path, title, nct_labels=None, class_names=None):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.figure(figsize=(8,6))

    if nct_labels is not None and class_names is not None:
        cmap = plt.get_cmap("tab10")
        for ci, cname in enumerate(class_names):
            m = (nct_labels == ci)
            if np.any(m):
                plt.scatter(
                    nct_2d[m, 0], nct_2d[m, 1],
                    alpha=0.55, s=10,
                    color=cmap(ci % 10),
                    label=cname
                )
    else:
        plt.scatter(nct_2d[:,0], nct_2d[:,1], alpha=0.5, s=10, label="NCT (Source)")

    plt.scatter(tcga_2d[:,0], tcga_2d[:,1], alpha=0.35, s=12, c="orange", label="TCGA (Target)")
    plt.legend(markerscale=2, fontsize=8, ncol=2, frameon=True)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

def umap_before_after(encoder, adapter_before, adapter_after, src_loader, tgt_loader, device, tag, class_names, umap_seed=42):
    # NOTE: UMAP visualization seed fixed (umap_seed) so plots are comparable across seeds.

    # --- FIX: deterministic shuffle so NCT doesn't look like only "ADI" due to folder-order bias ---
    gen = torch.Generator()
    gen.manual_seed(int(umap_seed))

    src_umap_loader = DataLoader(
        src_loader.dataset, batch_size=src_loader.batch_size,
        shuffle=True, generator=gen,
        num_workers=0, pin_memory=True, drop_last=False
    )
    tgt_umap_loader = DataLoader(
        tgt_loader.dataset, batch_size=tgt_loader.batch_size,
        shuffle=True, generator=gen,
        num_workers=0, pin_memory=True, drop_last=False
    )

    # BEFORE
    src_b, y_b = extract_features_for_umap(encoder, adapter_before, src_umap_loader, device, max_batches=40, return_labels=True)
    tgt_b = extract_features_for_umap(encoder, adapter_before, tgt_umap_loader, device, max_batches=40, return_labels=False)

    scaler, pca, um_ = fit_umap_projector(src_b, tgt_b, pca_dim=30, seed=umap_seed)
    src_b2 = project_umap(scaler, pca, um_, src_b)
    tgt_b2 = project_umap(scaler, pca, um_, tgt_b)

    before_path = os.path.join(OUT_DIR, f"umap_{tag}_before.png")
    plot_domain_gap(src_b2, tgt_b2, before_path, f"{tag} | BEFORE",
                    nct_labels=y_b, class_names=class_names)

    # AFTER (same projector)
    src_a = extract_features_for_umap(encoder, adapter_after, src_umap_loader, device, max_batches=40, return_labels=False)
    tgt_a = extract_features_for_umap(encoder, adapter_after, tgt_umap_loader, device, max_batches=40, return_labels=False)

    src_a2 = project_umap(scaler, pca, um_, src_a)
    tgt_a2 = project_umap(scaler, pca, um_, tgt_a)

    after_path = os.path.join(OUT_DIR, f"umap_{tag}_after.png")
    plot_domain_gap(src_a2, tgt_a2, after_path, f"{tag} | AFTER",
                    nct_labels=y_b, class_names=class_names)

    show_plot_from_file(before_path, title=f"{tag} | UMAP BEFORE", figsize=(7,6))
    show_plot_from_file(after_path,  title=f"{tag} | UMAP AFTER",  figsize=(7,6))

# =========================
# 9) TRAIN (B0-B4) + BEST CKPT + EARLY STOP  (OPTION A)
# =========================
def train_Bx_adapter(
    encoder, adapter, head,
    src_loader, tgt_loader, val7k_loader,
    device, mode="B0",
    epochs=10, lr=1e-3,
    lambda_p=1.0, lambda_c=0.1, lambda_m=0.05, lambda_s=0.1,
    proto_refresh_batches=200,
    patience=2,
    min_delta=1e-4,
    ramp_epochs=2,
    use_amp=True,
    score_alpha_mkmmd=0.25,
    score_beta_protodist=1.0,
    align_eval_every=1,
    align_num_batches=25,
    pad_eval_seed=42,          # <-- PAD classifier seed
    eta_label_prefix=""        # <-- to include "seed=.." in logs
):
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()

    for p in adapter.parameters(): p.requires_grad = True
    for p in head.parameters():    p.requires_grad = True

    opt = torch.optim.AdamW(
        list(adapter.parameters()) + list(head.parameters()),
        lr=lr, weight_decay=1e-4
    )

    scaler_amp = torch.amp.GradScaler('cuda', enabled=(use_amp and device.type == "cuda"))
    num_classes = head.fc.out_features

    protos = None
    if mode in ["B3", "B4"]:
        protos = compute_source_prototypes(
            encoder, adapter, src_loader, device,
            num_classes=num_classes, max_batches=proto_refresh_batches
        )

    history = {"total":[], "src":[], "proto":[], "coral":[], "mmd":[], "stain":[]}
    epoch_mean_losses = []
    epoch_logs = []

    best = {"score": -1e9, "epoch": 0, "adapter": None, "head": None, "metrics": None}
    bad_epochs = 0
    it_t = iter(tgt_loader)

    for ep in range(1, epochs+1):
        # ---- epoch start timing + peak memory reset ----
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t_ep0 = time.time()

        # ---- ETA print at epoch start (from previous epochs) ----
        if ep == 1:
            print(f"{eta_label_prefix}[{mode}] Ep {ep:02d} starting... (ETA after ep1)")
        else:
            prev_times = [row.get("epoch_time_sec") for row in epoch_logs if row.get("epoch_time_sec") is not None]
            if len(prev_times) > 0:
                avg_t = float(np.mean(prev_times))
                rem = (epochs - ep + 1)
                eta_sec = avg_t * rem
                print(f"{eta_label_prefix}[{mode}] Ep {ep:02d} starting... ~{eta_sec/60:.1f} min remaining (avg/ep {avg_t:.1f}s)")
            else:
                print(f"{eta_label_prefix}[{mode}] Ep {ep:02d} starting...")

        adapter.train(); head.train()
        ep_losses = []

        r = min(1.0, ep / float(max(1, ramp_epochs)))
        lc = lambda_c * r
        lm = lambda_m * r
        lp = lambda_p * r
        ls = lambda_s * r

        for xs01, ys, _ in tqdm(src_loader, desc=f"Train {mode} Ep{ep}/{epochs}", dynamic_ncols=True, mininterval=2):
            try:
                xt01, _ = next(it_t)
            except StopIteration:
                it_t = iter(tgt_loader)
                xt01, _ = next(it_t)

            xs01 = xs01.to(device, non_blocking=True)
            ys   = ys.to(device, non_blocking=True)
            xt01 = xt01.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type='cuda', enabled=(use_amp and device.type == "cuda")):
                fs = encoder(normalize_imnet(xs01))
                if isinstance(fs, (tuple, list)): fs = fs[0]
                fs = adapter(fs)
                logits_s = head(fs)
                loss_src = F.cross_entropy(logits_s, ys)

                total_loss = loss_src
                loss_proto = torch.tensor(0.0, device=device)
                loss_coral = torch.tensor(0.0, device=device)
                loss_mmd   = torch.tensor(0.0, device=device)
                loss_stain = torch.tensor(0.0, device=device)

                ft = None
                if mode in ["B1", "B2", "B3", "B4"]:
                    ft = encoder(normalize_imnet(xt01))
                    if isinstance(ft, (tuple, list)): ft = ft[0]
                    ft = adapter(ft)

                if mode in ["B1", "B2", "B3", "B4"]:
                    loss_coral = coral_loss(fs, ft)
                    total_loss = total_loss + lc * loss_coral

                if mode in ["B2", "B3", "B4"]:
                    loss_mmd = mkmmd_loss(fs, ft)
                    total_loss = total_loss + lm * loss_mmd

                if mode in ["B3", "B4"]:
                    loss_proto = proto_contrast_loss(fs, ys, protos)
                    total_loss = total_loss + lp * loss_proto

                if mode == "B4":
                    loss_stain = stain_consistency_loss(encoder, adapter, xt01)
                    total_loss = total_loss + ls * loss_stain

            scaler_amp.scale(total_loss).backward()
            scaler_amp.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(adapter.parameters()) + list(head.parameters()), max_norm=1.0)
            scaler_amp.step(opt)
            scaler_amp.update()

            history["total"].append(float(total_loss.item()))
            history["src"].append(float(loss_src.item()))
            history["proto"].append(float(loss_proto.item()))
            history["coral"].append(float(loss_coral.item()))
            history["mmd"].append(float(loss_mmd.item()))
            history["stain"].append(float(loss_stain.item()))
            ep_losses.append(float(total_loss.item()))

        if mode in ["B3", "B4"]:
            protos = compute_source_prototypes(
                encoder, adapter, src_loader, device,
                num_classes=num_classes, max_batches=proto_refresh_batches
            )

        mean_ep_loss = float(np.mean(ep_losses)) if len(ep_losses) > 0 else 0.0
        epoch_mean_losses.append(mean_ep_loss)

        macro_f1, bal_acc, acc = eval_7k(encoder, adapter, head, val7k_loader, device)

        coral_gap = mkmmd_gap = proto_dist = pad_gap = nc_entropy = majority = None
        if (ep % max(1, align_eval_every) == 0):
            coral_gap, mkmmd_gap, proto_dist, pad_gap, nc_entropy, majority = eval_tcga_alignment(
                encoder, adapter, src_loader, tgt_loader, device,
                num_classes=num_classes, num_batches=align_num_batches,
                knn_k=25, pad_max_samples=5000, pad_seed=pad_eval_seed
            )

        if coral_gap is None:
            score = float(macro_f1)
        else:
            score = float(macro_f1) - score_alpha_mkmmd * float(mkmmd_gap) - score_beta_protodist * float(proto_dist)

        ep_time = time.time() - t_ep0
        epoch_time_sec, max_gpu_mem_gb, ram_used_gb, gpu_util, gpu_mem_used_mb, gpu_mem_total_mb = get_resource_snapshot(ep_time)

        log_row = {
            "epoch": ep,
            "train_loss": mean_ep_loss,
            "macro_f1": float(macro_f1),
            "bal_acc": float(bal_acc),
            "acc": float(acc),
            "coral": None if coral_gap is None else float(coral_gap),
            "mkmmd": None if mkmmd_gap is None else float(mkmmd_gap),
            "protodist": None if proto_dist is None else float(proto_dist),
            "pad": None if pad_gap is None else float(pad_gap),
            "nc_entropy": None if nc_entropy is None else float(nc_entropy),
            "majority": None if majority is None else float(majority),
            "score": float(score),
            "ramp": float(r),
            "lp": float(lp), "lc": float(lc), "lm": float(lm), "ls": float(ls),
            "epoch_time_sec": float(epoch_time_sec),
            "max_gpu_mem_gb": float(max_gpu_mem_gb),
            "ram_used_gb": float(ram_used_gb),
            "gpu_util_pct": None if gpu_util is None else float(gpu_util),
            "gpu_mem_used_mb": None if gpu_mem_used_mb is None else float(gpu_mem_used_mb),
            "gpu_mem_total_mb": None if gpu_mem_total_mb is None else float(gpu_mem_total_mb),
        }
        epoch_logs.append(log_row)

        if coral_gap is None:
            print(f"{eta_label_prefix}[{mode}] Ep {ep:02d} | loss={mean_ep_loss:.4f} | 7K MacroF1={macro_f1:.4f} BalAcc={bal_acc:.4f} Acc={acc:.4f} | score={score:.4f} | time={epoch_time_sec:.1f}s | GPUutil={'N/A' if gpu_util is None else f'{gpu_util:.1f}%'}")
        else:
            print(f"{eta_label_prefix}[{mode}] Ep {ep:02d} | loss={mean_ep_loss:.4f} | 7K MacroF1={macro_f1:.4f} BalAcc={bal_acc:.4f} Acc={acc:.4f} | "
                  f"TCGA CORAL={coral_gap:.4f} MKMMD={mkmmd_gap:.4f} ProtoDist={proto_dist:.4f} "
                  f"PAD={pad_gap:.4f} NC-Ent={nc_entropy:.4f} Maj={majority:.4f} | score={score:.4f} | time={epoch_time_sec:.1f}s | peakGPU={max_gpu_mem_gb:.2f}GB | GPUutil={'N/A' if gpu_util is None else f'{gpu_util:.1f}%'}")

        if score > best["score"] + min_delta:
            best["score"] = score
            best["epoch"] = ep
            best["adapter"] = {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()}
            best["head"]    = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            best["metrics"] = log_row
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"{eta_label_prefix}[{mode}] Early stopping at epoch {ep} (best epoch={best['epoch']} best score={best['score']:.4f})")
                break

    if best["adapter"] is not None:
        adapter.load_state_dict({k: v.to(device) for k, v in best["adapter"].items()})
        head.load_state_dict({k: v.to(device) for k, v in best["head"].items()})

    return adapter, head, history, epoch_mean_losses, best, epoch_logs

    # =========================
    # 10) STATS: paired t-test (B0 vs B3)
    # =========================
    def paired_ttest_report(x, y, metric_name, out_lines, label_x="B0", label_y="B1"):
        """
        Safe paired t-test: never crashes.
        - Handles None/NaN
        - If scipy missing or n<2 => prints a safe note and returns
        """
        try:
            x = np.asarray([np.nan if v is None else v for v in x], dtype=np.float64)
            y = np.asarray([np.nan if v is None else v for v in y], dtype=np.float64)
    
            m = np.isfinite(x) & np.isfinite(y)
            x = x[m]; y = y[m]
    
            n = int(x.size)
            if n < 2:
                out_lines.append(
                    f"{metric_name}: {label_x} vs {label_y} | n={n} (need >=2 valid pairs) -> SKIPPED"
                )
                return
    
            mean_x = float(np.mean(x)); std_x = float(np.std(x, ddof=1))
            mean_y = float(np.mean(y)); std_y = float(np.std(y, ddof=1))
            diff = y - x
            mean_d = float(np.mean(diff)); std_d = float(np.std(diff, ddof=1))
    
            if ttest_rel is not None:
                t_stat, p_val = ttest_rel(y, x, nan_policy="omit")
                out_lines.append(
                    f"{metric_name}: {label_x} mean±std={mean_x:.4f}±{std_x:.4f} | "
                    f"{label_y} mean±std={mean_y:.4f}±{std_y:.4f} | "
                    f"Δ({label_y}-{label_x})={mean_d:.4f}±{std_d:.4f} | "
                    f"paired t={float(t_stat):.4f} p={float(p_val):.6g}"
                )
            else:
                # scipy yoksa sadece özet geç, patlamasın
                out_lines.append(
                    f"{metric_name}: {label_x} mean±std={mean_x:.4f}±{std_x:.4f} | "
                    f"{label_y} mean±std={mean_y:.4f}±{std_y:.4f} | "
                    f"Δ({label_y}-{label_x})={mean_d:.4f}±{std_d:.4f} | "
                    f"(scipy not found -> p-value not available)"
                )
    
        except Exception as e:
            out_lines.append(
                f"{metric_name}: {label_x} vs {label_y} | ERROR-BUT-SAFE: {type(e).__name__}: {e}"
            )
            return

# =========================
# 11) MAIN (Ablation/Final Profiles
# =========================
if __name__ == "__main__":
    # -----------------------------
    # CONFIG (profile-based)
    # -----------------------------
    EXPERIMENT_PROFILE = "final"  #final or ablation
    BATCH_SIZE = 64
    NUM_WORKERS = 0

    LR = 5e-4

    lambda_c = 0.05
    lambda_m = 0.05
    lambda_p = 0.3
    lambda_s = 0.1


    USE_AMP = True

    # profile -> seeds / tcga size / epochs / early-stop behavior
    if EXPERIMENT_PROFILE.lower() == "ablation":
        print("Ablation Mod working on")
        SEEDS = [0, 1, 2]         # ablation: 3 seeds
        EPOCHS = 5                # ablation: short
        tcga_max_samples = 10000  # ablation: mini tcga
        RAMP_EPOCHS = 2

        # ---- Disable early stopping for ablation ----
        PATIENCE = 10**9          # effectively off (won't trigger within 5 epochs)
        MIN_DELTA = 0.0           # treat any improvement as improvement (best tracking)
        ES_TAG = "NO EARLY STOP"
    else:
        print("Final Mod working on")
        SEEDS = [0, 1, 2, 3, 4]   # final: 5 seeds
        EPOCHS = 10               # final: max epochs, early stop applies
        tcga_max_samples = 30000   # final: If is None; works full tcga

        # ---- Enable early stopping for final ----
        PATIENCE = 4
        MIN_DELTA = 1e-4
        RAMP_EPOCHS = 4
        ES_TAG = "EARLY STOP"

    # UMAP only on a single seed to save time
    UMAP_ONLY_SEED = SEEDS[0]     # first seed only

    # fixed TCGA list across seeds/runs (DATA seed)
    TCGA_DATA_SEED = 42
    fixed_tcga_paths = make_fixed_tcga_paths(
        TCGA_PATH, max_samples=tcga_max_samples, seed=TCGA_DATA_SEED
    )
    print(f"[TCGA] Fixed sample set size: {len(fixed_tcga_paths)} (seed={TCGA_DATA_SEED}, max_samples={tcga_max_samples})")
    print(f"[UMAP] Only plotting UMAP for seed={UMAP_ONLY_SEED}")
    print(f"[STOP] Profile={EXPERIMENT_PROFILE} | {ES_TAG} | patience={PATIENCE} min_delta={MIN_DELTA}")

    MODE_DESC = {
        "B0": "CE (baseline)",
        "B1": "CE + CORAL",
        "B2": "CE + CORAL + MKMMD",
        "B3": "CE + CORAL + MKMMD + Proto",
        "B4": "CE + CORAL + MKMMD + Proto + StainCons"
    }

    MODES = ["B0", "B1", "B2", "B3", "B4"]

    # Frozen encoder (create once)
    encoder = load_uni2h_encoder().to(device)
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False

    # CSV log (seed-level results)
    csv_path = os.path.join(OUT_DIR, "seed_results_ALL.csv")
    csv_fields = [
        "seed","mode","best_epoch","best_score",
        "7K_macroF1","7K_balAcc","7K_acc",
        "TCGA_CORAL","TCGA_MKMMD","TCGA_ProtoDist","TCGA_PAD","TCGA_NCEntropy","TCGA_Majority",
        "train_total_time_sec","train_avg_epoch_sec","peak_gpu_mem_gb","peak_ram_used_gb",
        "ckpt_path"
    ]
    with open(csv_path, "w", newline="", encoding="utf-8") as fcsv:
        writer = csv.DictWriter(fcsv, fieldnames=csv_fields)
        writer.writeheader()

    # store for paired t-test
    all_seed_results = {}  # seed -> mode -> metrics dict

    # -----------------------------
    # SEED LOOP
    # -----------------------------
    for seed in SEEDS:
        print("\n" + "#"*100)
        print(f"SEED RUN: {seed} (profile={EXPERIMENT_PROFILE}, TCGA fixed seed={TCGA_DATA_SEED}, n_tcga={len(fixed_tcga_paths)})")
        print("#"*100)

        set_seed(seed)

        # rebuild loaders so shuffle order follows the seed
        src_loader, val7k_loader, tgt_loader, classes = make_loaders(
            batch_size=BATCH_SIZE,
            num_workers_train=NUM_WORKERS,
            num_workers_eval=0,
            tcga_max_samples=None,            # IMPORTANT: fixed list already chosen
            fixed_tcga_paths=fixed_tcga_paths
        )
        num_classes = len(classes)
        print("Classes:", classes, "| num_classes:", num_classes)

        all_seed_results[seed] = {}

        for mode in MODES:
            print("\n" + "="*90)
            print(f"RUN: seed={seed} | {mode} | Frozen UNI2-H + Adapter+Head (Option A + BEST CKPT + {ES_TAG})")
            print("="*90)

            adapter = BottleneckAdapter(dim=1536, reduction=16).to(device)
            head = LinearHead(in_dim=1536, num_classes=num_classes).to(device)

            # UMAP BEFORE (same random init)
            adapter_before = BottleneckAdapter(dim=1536, reduction=16).to(device)
            adapter_before.load_state_dict(adapter.state_dict())

            adapter, head, history, epoch_mean_losses, best, epoch_logs = train_Bx_adapter(
                encoder, adapter, head,
                src_loader, tgt_loader, val7k_loader,
                device=device, mode=mode,
                epochs=EPOCHS, lr=LR,
                lambda_p=lambda_p, lambda_c=lambda_c, lambda_m=lambda_m, lambda_s=lambda_s,
                patience=PATIENCE,
                min_delta=MIN_DELTA,          # <-- profile-controlled
                ramp_epochs=RAMP_EPOCHS,
                use_amp=USE_AMP,
                align_eval_every=1,
                align_num_batches=25,
                pad_eval_seed=seed,           # PAD domain classifier uses current training seed
                eta_label_prefix=f"[seed={seed}] "
            )

            # Plots
            loss_path = os.path.join(OUT_DIR, f"loss_curve_seed{seed}_{mode}.png")
            plot_loss_curves(history, out_path=loss_path, title=f"seed{seed} {mode} | Loss Curves (step-level)")

            epoch_loss_path = os.path.join(OUT_DIR, f"epoch_loss_curve_seed{seed}_{mode}.png")
            plot_epoch_loss_curve(epoch_mean_losses, out_path=epoch_loss_path, title=f"seed{seed} {mode} | Mean Train Loss (epoch-level)")

            # Final metrics (best weights already loaded)
            macro_f1, bal_acc, acc = eval_7k(encoder, adapter, head, val7k_loader, device)

            coral_gap, mkmmd_gap, proto_dist, pad_gap, nc_entropy, majority = eval_tcga_alignment(
                encoder, adapter, src_loader, tgt_loader, device,
                num_classes=num_classes, num_batches=50,
                knn_k=25, pad_max_samples=5000, pad_seed=seed
            )

            # UMAP (only once per profile to save time)
            if seed == UMAP_ONLY_SEED:
                umap_before_after(
                    encoder, adapter_before, adapter,
                    src_loader, tgt_loader, device,
                    tag=f"seed{seed}_{mode}",
                    class_names=classes,
                    umap_seed=42
                )

            # Save checkpoint (best)
            ckpt_path = os.path.join(OUT_DIR, f"ckpt_seed{seed}_{mode}.pth")
            torch.save({
                "seed": int(seed),
                "mode": mode,
                "classes": classes,
                "best_epoch": int(best["epoch"]),
                "best_score": float(best["score"]),
                "adapter_state": adapter.state_dict(),
                "head_state": head.state_dict(),
                "best_metrics": best["metrics"],
                "epoch_logs": epoch_logs,
                "metrics": {
                    "7K_macroF1": float(macro_f1),
                    "7K_balAcc": float(bal_acc),
                    "7K_acc": float(acc),
                    "TCGA_CORAL": float(coral_gap),
                    "TCGA_MKMMD": float(mkmmd_gap),
                    "TCGA_ProtoDist": float(proto_dist),
                    "TCGA_PAD": float(pad_gap),
                    "TCGA_NCEntropy": float(nc_entropy),
                    "TCGA_Majority": float(majority),
                }
            }, ckpt_path)

            # Resource summary (from epoch logs)
            total_time = float(np.sum([r["epoch_time_sec"] for r in epoch_logs if r.get("epoch_time_sec") is not None]))
            avg_time   = float(np.mean([r["epoch_time_sec"] for r in epoch_logs if r.get("epoch_time_sec") is not None])) if len(epoch_logs) > 0 else 0.0
            peak_gpu   = float(np.max([r["max_gpu_mem_gb"] for r in epoch_logs if r.get("max_gpu_mem_gb") is not None])) if len(epoch_logs) > 0 else 0.0
            peak_ram   = float(np.max([r["ram_used_gb"] for r in epoch_logs if r.get("ram_used_gb") is not None])) if len(epoch_logs) > 0 else 0.0

            row = {
                "seed": int(seed),
                "mode": mode,
                "best_epoch": int(best["epoch"]),
                "best_score": float(best["score"]),
                "7K_macroF1": float(macro_f1),
                "7K_balAcc": float(bal_acc),
                "7K_acc": float(acc),
                "TCGA_CORAL": float(coral_gap),
                "TCGA_MKMMD": float(mkmmd_gap),
                "TCGA_ProtoDist": float(proto_dist),
                "TCGA_PAD": float(pad_gap),
                "TCGA_NCEntropy": float(nc_entropy),
                "TCGA_Majority": float(majority),
                "train_total_time_sec": float(total_time),
                "train_avg_epoch_sec": float(avg_time),
                "peak_gpu_mem_gb": float(peak_gpu),
                "peak_ram_used_gb": float(peak_ram),
                "ckpt_path": ckpt_path
            }

            all_seed_results[seed][mode] = dict(row)

            with open(csv_path, "a", newline="", encoding="utf-8") as fcsv:
                writer = csv.DictWriter(fcsv, fieldnames=csv_fields)
                writer.writerow(row)

            print(
                f"[seed={seed} {mode}] FINAL(best) "
                f"7K MacroF1={macro_f1:.4f} BalAcc={bal_acc:.4f} Acc={acc:.4f} | "
                f"TCGA CORAL={coral_gap:.4f} MKMMD={mkmmd_gap:.4f} ProtoDist={proto_dist:.4f} | "
                f"PAD={pad_gap:.4f} NC-Ent={nc_entropy:.4f} Maj={majority:.4f} | "
                f"TrainTime={total_time/60:.1f}min PeakGPU={peak_gpu:.2f}GB"
            )

        # per-seed summary txt
        seed_summary_path = os.path.join(OUT_DIR, f"results_summary_seed{seed}.txt")
        with open(seed_summary_path, "w", encoding="utf-8") as f:
            f.write("OPTION A ABLATION SUMMARY (+ PAD + NC-Entropy + Resources)\n")
            f.write(f"PROFILE={EXPERIMENT_PROFILE}\n")
            f.write(f"SEED={seed}\n")
            f.write(f"lambda_c={lambda_c} lambda_m={lambda_m} lambda_p={lambda_p} lambda_s={lambda_s}\n")
            f.write(f"epochs={EPOCHS} lr={LR} batch={BATCH_SIZE} ramp_epochs={RAMP_EPOCHS} patience={PATIENCE} min_delta={MIN_DELTA} amp={USE_AMP}\n")
            f.write(f"fixed_tcga_size={len(fixed_tcga_paths)} tcga_data_seed={TCGA_DATA_SEED} tcga_max_samples={tcga_max_samples}\n\n")
            for m in MODES:
                d = all_seed_results[seed][m]
                desc = MODE_DESC.get(m, "-")
                f.write(
                    f"{m} ({desc}) | best_epoch={d['best_epoch']} best_score={d['best_score']:.4f} | "
                    f"7K_macroF1={d['7K_macroF1']:.4f} 7K_balAcc={d['7K_balAcc']:.4f} 7K_acc={d['7K_acc']:.4f} | "
                    f"TCGA_CORAL={d['TCGA_CORAL']:.4f} TCGA_MKMMD={d['TCGA_MKMMD']:.4f} "
                    f"TCGA_ProtoDist={d['TCGA_ProtoDist']:.4f} PAD={d['TCGA_PAD']:.4f} "
                    f"NCEnt={d['TCGA_NCEntropy']:.4f} Maj={d['TCGA_Majority']:.4f} | "
                    f"time_total={d['train_total_time_sec']:.1f}s time_avg_ep={d['train_avg_epoch_sec']:.1f}s "
                    f"peakGPU={d['peak_gpu_mem_gb']:.2f}GB peakRAM={d['peak_ram_used_gb']:.2f}GB\n"
                )

    # -----------------------------
    # PAIRED T-TEST: FINAL profile only
    # Only 7K_macroF1, baseline vs all modes
    # -----------------------------
    if EXPERIMENT_PROFILE.lower() == "final":
        BASELINE = "B0"
        COMPARE_MODES = ["B1", "B2", "B3", "B4"]
    
        lines = []
        lines.append("PAIRED T-TEST (Macro-F1 only) Baseline vs All Modes across seeds\n")
        lines.append(f"PROFILE={EXPERIMENT_PROFILE}\n")
        lines.append(f"SEEDS={SEEDS}\n")
        lines.append(f"BASELINE={BASELINE} COMPARE={COMPARE_MODES}\n")
        if ttest_rel is None:
            lines.append("NOTE: scipy not found -> p-values not available.\n")
        lines.append("------------------------------------------------------------------")
    
        if len(SEEDS) >= 2:
            for m in COMPARE_MODES:
                lines.append(f"\n### {m} vs {BASELINE}")
    
                x = [all_seed_results.get(s, {}).get(BASELINE, {}).get("7K_macroF1", np.nan) for s in SEEDS]
                y = [all_seed_results.get(s, {}).get(m, {}).get("7K_macroF1", np.nan) for s in SEEDS]
    
                paired_ttest_report(
                    x, y,
                    metric_name="7K Macro-F1",
                    out_lines=lines,
                    label_x=BASELINE,
                    label_y=m
                )
        else:
            lines.append("Only 1 seed -> paired t-test skipped (need >=2 seeds).")
    
        ttest_path = os.path.join(OUT_DIR, f"paired_ttest_{BASELINE}_vs_all_macroF1.txt")
        with open(ttest_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    else:
        ttest_path = None


    # -----------------------------
    # Final overall summary (always)
    # -----------------------------
    final_summary_path = os.path.join(OUT_DIR, "results_summary_ALL_SEEDS.txt")
    with open(final_summary_path, "w", encoding="utf-8") as f:
        f.write("ALL SEEDS SUMMARY\n")
        f.write(f"PROFILE={EXPERIMENT_PROFILE}\n")
        f.write(f"SEEDS={SEEDS}\n")
        f.write(f"fixed_tcga_size={len(fixed_tcga_paths)} tcga_data_seed={TCGA_DATA_SEED} tcga_max_samples={tcga_max_samples}\n")
        f.write(f"CSV={csv_path}\n")
        if ttest_path is not None:
            f.write(f"Paired t-test={ttest_path}\n")
        else:
            f.write("Paired t-test=SKIPPED (ablation profile)\n")

    print("\nExperiments finished.")
    print("Outputs:", OUT_DIR)
    print("Seed CSV:", csv_path)
    if ttest_path is not None:
        print("Paired t-test:", ttest_path)
    else:
        print("Paired t-test: SKIPPED (ablation profile)")
    print("All-seeds summary:", final_summary_path)





Final Mod working on
[TCGA] Fixed sample set size: 30000 (seed=42, max_samples=30000)
[UMAP] Only plotting UMAP for seed=0
[STOP] Profile=final | EARLY STOP | patience=4 min_delta=0.0001

####################################################################################################
SEED RUN: 0 (profile=final, TCGA fixed seed=42, n_tcga=30000)
####################################################################################################
Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM'] | num_classes: 9

RUN: seed=0 | B0 | Frozen UNI2-H + Adapter+Head (Option A + BEST CKPT + EARLY STOP)
[seed=0] [B0] Ep 01 starting... (ETA after ep1)


Eval 7K: 100%|██████████| 113/113 [02:03<00:00,  1.09s/it]


[seed=0] [B0] Ep 01 | loss=0.0273 | 7K MacroF1=0.9488 BalAcc=0.9524 Acc=0.9609 | TCGA CORAL=0.0457 MKMMD=0.3496 ProtoDist=0.5940 PAD=2.0000 NC-Ent=0.0683 Maj=0.9340 | score=0.2674 | time=2126.4s | peakGPU=6.93GB | GPUutil=0.0%
[seed=0] [B0] Ep 02 starting... ~319.0 min remaining (avg/ep 2126.4s)


Eval 7K: 100%|██████████| 113/113 [02:03<00:00,  1.10s/it]


[seed=0] [B0] Ep 02 | loss=0.0059 | 7K MacroF1=0.9500 BalAcc=0.9558 Acc=0.9639 | TCGA CORAL=0.0710 MKMMD=0.3483 ProtoDist=0.5759 PAD=2.0000 NC-Ent=0.0662 Maj=0.9393 | score=0.2871 | time=2147.8s | peakGPU=6.93GB | GPUutil=0.0%
[seed=0] [B0] Ep 03 starting... ~284.9 min remaining (avg/ep 2137.1s)


...

Eval 7K: 100%|██████████| 113/113 [02:03<00:00,  1.09s/it]


[seed=0] [B0] Ep 06 | loss=0.0022 | 7K MacroF1=0.8922 BalAcc=0.9099 Acc=0.8936 | TCGA CORAL=0.2401 MKMMD=0.3117 ProtoDist=0.4776 PAD=1.9958 NC-Ent=0.0452 Maj=0.9582 | score=0.3366 | time=2098.2s | peakGPU=6.93GB | GPUutil=0.0%
[seed=0] [B0] Ep 07 starting... ~141.9 min remaining (avg/ep 2128.8s)


Eval 7K: 100%|██████████| 113/113 [02:06<00:00,  1.12s/it]


[seed=0] [B0] Ep 07 | loss=0.0020 | 7K MacroF1=0.9194 BalAcc=0.9309 Acc=0.9288 | TCGA CORAL=0.4188 MKMMD=0.2928 ProtoDist=0.4440 PAD=1.9958 NC-Ent=0.0453 Maj=0.9565 | score=0.4022 | time=2152.7s | peakGPU=6.93GB | GPUutil=0.0%
[seed=0] [B0] Ep 08 starting... ~106.6 min remaining (avg/ep 2132.3s)


Train B0 Ep8/10:   5%|▌         | 82/1562 [01:32<28:14,  1.14s/it]